In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql import DataFrame
import pyspark.sql.functions as F   
import re
from functools import reduce
from operator import add
import glob
import os
import shutil


# Create a spark session
spark = (
    SparkSession.builder.appName("MAST30034 Assignment1")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config('spark.driver.memory', '4g')
    .config('spark.executor.memory', '2g')
    .getOrCreate()
)

In [2]:
def print_shape(df: DataFrame, name: str = "DataFrame") -> None:
    """Helper function to print shape of dataframe with its name."""
    row_count = df.count()
    col_count = len(df.columns)
    print(f"{name} shape: ({row_count}, {col_count})")

In [3]:
# Reading the taxi datasets
# shape: (802387, 20)
sdf_green = spark.read.parquet('../../data/raw/tlc_data/green')
print_shape(sdf_green, "sdf_green")
# shape: (67721884, 19)
sdf_yellow = spark.read.parquet('../../data/raw/tlc_data/yellow')
print_shape(sdf_yellow, "sdf_yellow")
# shape: (349585798, 25)
sdf_hv = spark.read.parquet('../../data/raw/tlc_data/high_volume')   
print_shape(sdf_hv, "sdf_hv")                 

sdf_green shape: (802387, 21)
sdf_yellow shape: (67721884, 20)
sdf_hv shape: (349585798, 25)


In [4]:
"""
Formatting the columns
"""
def to_snake_case(column_name: str) -> str:
    """Convert a column name to snake_case (e.g. "OnStreetName" -> "on_street_name")."""
    column_name = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1_\2", column_name)
    column_name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", column_name)
    return (
        column_name.strip()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
        .lower()
    )
    
def rename_columns_to_snake_case(df: DataFrame) -> DataFrame:
    """Applies to_snake_case to all columns in a PySpark DataFrame."""
    for col in df.columns:
        df = df.withColumnRenamed(col, to_snake_case(col))
    return df

# snake case all columns
sdf_green = rename_columns_to_snake_case(sdf_green) # green taxis
sdf_yellow = rename_columns_to_snake_case(sdf_yellow) # yellow taxis
sdf_hv= rename_columns_to_snake_case(sdf_hv) #high volume/ high frequency

# Add the taxi type column to each DataFrame
# shape: (802387, 21) -> (802387, 22)
sdf_green = sdf_green.withColumn("taxi_type", F.lit("green"))
print_shape(sdf_green, "sdf_green")
# shape: (67721884, 20) -> (67721884, 21)
sdf_yellow = sdf_yellow.withColumn("taxi_type", F.lit("yellow"))
print_shape(sdf_yellow, "sdf_yellow")
# shape: (349585798, 25) -> (349585798, 26)
sdf_hv = sdf_hv.withColumn("taxi_type", F.lit("hv"))
print_shape(sdf_hv, "sdf_hv")

sdf_green shape: (802387, 22)
sdf_yellow shape: (67721884, 21)
sdf_hv shape: (349585798, 26)


In [5]:
#uncomment to check schemas 
# sdf_green.printSchema()
# sdf_yellow.printSchema()
# sdf_hv.printSchema()

In [6]:
# Data Cleaning

# ==========================================
# 1. SDF_GREEN CLEANING
# ==========================================

# 2025 pickups
# shape: (802387, 22) -> (802377, 22)
sdf_green = sdf_green.filter(F.year(F.col("lpep_pickup_datetime")) >= 2025)
print_shape(sdf_green, "sdf_green after scope filter (2025+)")

# Valid start date before end date
# shape: (802377, 22) -> (801031, 22)
sdf_green = sdf_green.filter(
    (F.col("lpep_dropoff_datetime") >= F.col("lpep_pickup_datetime")) |
    F.col("lpep_pickup_datetime").isNull() |
    F.col("lpep_dropoff_datetime").isNull()
)
print_shape(sdf_green, "sdf_green after date validity filter")

# Valid pickup location IDs — 1-263
# shape: (801031, 22) -> (798464, 22)
sdf_green = sdf_green.filter(
    F.col("PULocationID").between(1, 263) |
    F.col("PULocationID").isNull()
)
print_shape(sdf_green, "sdf_green after PU location validity filter")

# Valid dropoff location IDs — 1-263
# shape: (798464, 22) -> (788245, 22)
sdf_green = sdf_green.filter(
    F.col("DOLocationID").between(1, 263) |
    F.col("DOLocationID").isNull()
)
print_shape(sdf_green, "sdf_green after DO location validity filter")

# Valid trip distance — 0.1 to 150 miles
# shape: (788245, 22) -> (755933, 22)
sdf_green = sdf_green.filter(
    (F.col("trip_distance") >= 0.1) & (F.col("trip_distance") < 150)
)
print_shape(sdf_green, "sdf_green after trip_distance validity filter")

# Valid trip duration — 60 seconds to 4 hours (14400s)
# shape: (755933, 22) -> (747638, 22)
sdf_green = sdf_green.filter(
    (F.unix_timestamp("lpep_dropoff_datetime") - F.unix_timestamp("lpep_pickup_datetime")).between(60, 14400)
)
print_shape(sdf_green, "sdf_green after trip_duration validity filter")

# Valid passenger count — non-zero and non-negative (sdf_green)
# shape: (747638, 22) -> (737864, 22)
sdf_green = sdf_green.filter(
    (F.col("passenger_count") > 0) | F.col("passenger_count").isNull()
)
print_shape(sdf_green, "sdf_green after passenger_count validity filter")

# Valid monetary amounts — non-negative fees (sdf_green)
# shape: (737864, 22) -> (737214, 22)
sdf_green = sdf_green.filter(
    ((F.col("fare_amount") >= 0) | F.col("fare_amount").isNull()) &
    ((F.col("extra") >= 0) | F.col("extra").isNull()) &
    ((F.col("mta_tax") >= 0) | F.col("mta_tax").isNull()) &
    ((F.col("tip_amount") >= 0) | F.col("tip_amount").isNull()) &
    ((F.col("tolls_amount") >= 0) | F.col("tolls_amount").isNull()) &
    ((F.col("ehail_fee") >= 0) | F.col("ehail_fee").isNull()) &
    ((F.col("improvement_surcharge") >= 0) | F.col("improvement_surcharge").isNull()) &
    ((F.col("total_amount") >= 0) | F.col("total_amount").isNull()) &
    ((F.col("congestion_surcharge") >= 0) | F.col("congestion_surcharge").isNull()) &
    ((F.col("cbd_congestion_fee") >= 0) | F.col("cbd_congestion_fee").isNull())
)
print_shape(sdf_green, "sdf_green after fee validity filter")


# ==========================================
# 2. SDF_YELLOW CLEANING
# ==========================================

# 2025 pickups
# shape: (67721884, 21) -> (67721847, 21)
sdf_yellow = sdf_yellow.filter(F.year(F.col("tpep_pickup_datetime")) >= 2025)
print_shape(sdf_yellow, "sdf_yellow after scope filter (2025+)")

# Valid start date before end date
# shape: (67721847, 21) -> (67719607, 21)
sdf_yellow = sdf_yellow.filter(
    (F.col("tpep_dropoff_datetime") >= F.col("tpep_pickup_datetime")) |
    F.col("tpep_pickup_datetime").isNull() |
    F.col("tpep_dropoff_datetime").isNull()
)
print_shape(sdf_yellow, "sdf_yellow after date validity filter")

# Valid pickup location IDs — 1-263
# shape: (67719607, 21) -> (67586847, 21)
sdf_yellow = sdf_yellow.filter(
    F.col("PULocationID").between(1, 263) |
    F.col("PULocationID").isNull()
)
print_shape(sdf_yellow, "sdf_yellow after PU location validity filter")

# Valid dropoff location IDs — 1-263
# shape: (67586847, 21) -> (67228085, 21)
sdf_yellow = sdf_yellow.filter(
    F.col("DOLocationID").between(1, 263) |
    F.col("DOLocationID").isNull()
)
print_shape(sdf_yellow, "sdf_yellow after DO location validity filter")

# Valid trip distance — 0.1 to 150 miles
# shape: (67228085, 21) -> (64759473, 21)
sdf_yellow = sdf_yellow.filter(
    (F.col("trip_distance") >= 0.1) & (F.col("trip_distance") < 150)
)
print_shape(sdf_yellow, "sdf_yellow after trip_distance validity filter")

# Valid trip duration — 60 seconds to 4 hours (14400s)
# shape: (64759473, 21) -> (63920694, 21)
sdf_yellow = sdf_yellow.filter(
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")).between(60, 14400)
)
print_shape(sdf_yellow, "sdf_yellow after trip_duration validity filter")

# Valid passenger count — non-zero and non-negative (sdf_yellow)
# shape: (63920694, 21) -> (63615605, 21)
sdf_yellow = sdf_yellow.filter(
    (F.col("passenger_count") > 0) | F.col("passenger_count").isNull()
)
print_shape(sdf_yellow, "sdf_yellow after passenger_count validity filter")

# Valid monetary amounts — non-negative fees (sdf_yellow)
# shape: (63615605, 21) -> (61018058, 21)
sdf_yellow = sdf_yellow.filter(
    ((F.col("fare_amount") >= 0) | F.col("fare_amount").isNull()) &
    ((F.col("extra") >= 0) | F.col("extra").isNull()) &
    ((F.col("mta_tax") >= 0) | F.col("mta_tax").isNull()) &
    ((F.col("tip_amount") >= 0) | F.col("tip_amount").isNull()) &
    ((F.col("tolls_amount") >= 0) | F.col("tolls_amount").isNull()) &
    ((F.col("improvement_surcharge") >= 0) | F.col("improvement_surcharge").isNull()) &
    ((F.col("total_amount") >= 0) | F.col("total_amount").isNull()) &
    ((F.col("congestion_surcharge") >= 0) | F.col("congestion_surcharge").isNull()) &
    ((F.col("Airport_fee") >= 0) | F.col("Airport_fee").isNull()) &
    ((F.col("cbd_congestion_fee") >= 0) | F.col("cbd_congestion_fee").isNull())
)
print_shape(sdf_yellow, "sdf_yellow after fee validity filter")


# ==========================================
# 3. SDF_HV CLEANING
# ==========================================

# 2025 pickups
# shape: (349585798, 26) -> (349585798, 26)
sdf_hv = sdf_hv.filter(F.year(F.col("pickup_datetime")) >= 2025)
print_shape(sdf_hv, "sdf_hv after scope filter (2025+)")

# Valid start date before end date
# shape: (349585798, 26) -> (349573594, 26)
sdf_hv = sdf_hv.filter(
    (F.col("dropoff_datetime") >= F.col("pickup_datetime")) |
    F.col("pickup_datetime").isNull() |
    F.col("dropoff_datetime").isNull()
)
print_shape(sdf_hv, "sdf_hv after date validity filter")

# Valid pickup location IDs — 1-263
# shape: (349573594, 26) -> (349557873, 26)
sdf_hv = sdf_hv.filter(
    F.col("PULocationID").between(1, 263) |
    F.col("PULocationID").isNull()
)
print_shape(sdf_hv, "sdf_hv after PU location validity filter")

# Valid dropoff location IDs — 1-263
# shape: (349557873, 26) -> (333923627, 26)
sdf_hv = sdf_hv.filter(
    F.col("DOLocationID").between(1, 263) |
    F.col("DOLocationID").isNull()
)
print_shape(sdf_hv, "sdf_hv after DO location validity filter")

# Valid trip distance — 0.1 to 150 miles
# shape: (333923627, 26) -> (333803264, 26)
sdf_hv = sdf_hv.filter(
    (F.col("trip_miles") >= 0.1) & (F.col("trip_miles") < 150)
)
print_shape(sdf_hv, "sdf_hv after trip_miles validity filter")

# Valid trip duration — 60 seconds to 4 hours (trip_time is already an integer in sdf_hv)
# shape: (333803264, 26) -> (333788283, 26)
sdf_hv = sdf_hv.filter(
    F.col("trip_time").between(60, 14400)
)
print_shape(sdf_hv, "sdf_hv after trip_time validity filter")

# Valid monetary amounts — non-negative fees (sdf_hv)
# shape: (333788283, 26) -> (333726631, 26)
sdf_hv = sdf_hv.filter(
    ((F.col("base_passenger_fare") >= 0) | F.col("base_passenger_fare").isNull()) &
    ((F.col("tolls") >= 0) | F.col("tolls").isNull()) &
    ((F.col("bcf") >= 0) | F.col("bcf").isNull()) &
    ((F.col("sales_tax") >= 0) | F.col("sales_tax").isNull()) &
    ((F.col("congestion_surcharge") >= 0) | F.col("congestion_surcharge").isNull()) &
    ((F.col("airport_fee") >= 0) | F.col("airport_fee").isNull()) &
    ((F.col("tips") >= 0) | F.col("tips").isNull()) &
    ((F.col("driver_pay") >= 0) | F.col("driver_pay").isNull()) &
    ((F.col("cbd_congestion_fee") >= 0) | F.col("cbd_congestion_fee").isNull())
)
print_shape(sdf_hv, "sdf_hv after fee validity filter")

sdf_green after scope filter (2025+) shape: (802377, 22)
sdf_green after date validity filter shape: (801031, 22)
sdf_green after PU location validity filter shape: (798464, 22)
sdf_green after DO location validity filter shape: (788245, 22)
sdf_green after trip_distance validity filter shape: (755933, 22)
sdf_green after trip_duration validity filter shape: (747638, 22)
sdf_green after passenger_count validity filter shape: (737864, 22)
sdf_green after fee validity filter shape: (737214, 22)
sdf_yellow after scope filter (2025+) shape: (67721847, 21)
sdf_yellow after date validity filter shape: (67719607, 21)
sdf_yellow after PU location validity filter shape: (67586847, 21)
sdf_yellow after DO location validity filter shape: (67228085, 21)
sdf_yellow after trip_distance validity filter shape: (64759473, 21)
sdf_yellow after trip_duration validity filter shape: (63920694, 21)
sdf_yellow after passenger_count validity filter shape: (63615605, 21)
sdf_yellow after fee validity filter sh

In [7]:
#select only relevant columns
selected_columns = ['dropoff_datetime', 'pickup_datetime', 'pu_location_id', 'trip_duration', 'total_amount', 'taxi_type']

# Green 
# Rename the datetime columns to match each other
sdf_green = (sdf_green.withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')
                    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime'))
# calculate trip duration
# shape: (737214, 22) -> (737214, 23)
sdf_green = sdf_green \
    .withColumn(
        "trip_duration", 
        F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")
    )
print_shape(sdf_green, "sdf_green adding trip duration")
# select columns
# shape: (737214, 23) -> (737214, 6)
sdf_green = sdf_green.select(*selected_columns)
print_shape(sdf_green, "sdf_green after selection")

#Yellow
# Rename the datetime columns to match each other
sdf_yellow = (sdf_yellow.withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')
                    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime'))
# calculate trip duration
# shape: (61018058, 21) -> (61018058, 22)
sdf_yellow = sdf_yellow \
    .withColumn(
        "trip_duration", 
        F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")
    )
print_shape(sdf_yellow, "sdf_green adding trip duration")
# select columns
# shape: (61018058, 22) -> (61018058, 6)
sdf_yellow = sdf_yellow.select(*selected_columns)
print_shape(sdf_yellow, "sdf_yellow after selection")

# HV
# Rename the datetime columns to match each other
sdf_hv = (sdf_hv.withColumnRenamed('trip_miles', 'trip_distance')
                    .withColumnRenamed('trip_time', 'trip_duration'))
#calculate the total pay for hv trips
fare_cols = [
    "base_passenger_fare", "tolls", "bcf", "sales_tax",
    "congestion_surcharge", "airport_fee", "tips", "cbd_congestion_fee",
]
# shape: (333726631, 26) -> (333726631, 27)
sdf_hv = sdf_hv.withColumn(
    "total_amount",
    reduce(add, [F.coalesce(F.col(c), F.lit(0)) for c in fare_cols])
)
print_shape(sdf_hv, "sdf_hv")
# shape: (333726631, 27) -> (333726631, 6)
sdf_hv = sdf_hv.select(*selected_columns)
print_shape(sdf_hv, "sdf_yellow after selection")


sdf_green adding trip duration shape: (737214, 23)
sdf_green after selection shape: (737214, 6)
sdf_green adding trip duration shape: (61018058, 22)
sdf_yellow after selection shape: (61018058, 6)
sdf_hv shape: (333726631, 27)
sdf_yellow after selection shape: (333726631, 6)


In [8]:
print("--- Taxi Null Counts ---")
sdf_green.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in sdf_green.columns]).show()
sdf_yellow.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in sdf_yellow.columns]).show()
sdf_hv.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in sdf_hv.columns]).show()
# No imputation needed!

--- Taxi Null Counts ---
+----------------+---------------+--------------+-------------+------------+---------+
|dropoff_datetime|pickup_datetime|pu_location_id|trip_duration|total_amount|taxi_type|
+----------------+---------------+--------------+-------------+------------+---------+
|               0|              0|             0|            0|           0|        0|
+----------------+---------------+--------------+-------------+------------+---------+

+----------------+---------------+--------------+-------------+------------+---------+
|dropoff_datetime|pickup_datetime|pu_location_id|trip_duration|total_amount|taxi_type|
+----------------+---------------+--------------+-------------+------------+---------+
|               0|              0|             0|            0|           0|        0|
+----------------+---------------+--------------+-------------+------------+---------+

+----------------+---------------+--------------+-------------+------------+---------+
|dropoff_datetim

In [9]:
def compare_schemas(df1, df2, name1="DF1", name2="DF2"):
    """Compares names and data types between two DataFrames."""
    types1 = dict(df1.dtypes)
    types2 = dict(df2.dtypes)
    
    all_cols = sorted(list(set(types1.keys()).union(set(types2.keys()))))
    
    mismatches = []
    for col in all_cols:
        t1 = types1.get(col, "MISSING")
        t2 = types2.get(col, "MISSING")
        if t1 != t2:
            mismatches.append((col, t1, t2))
            
    if not mismatches:
        print("Schemas match!")
    else:
        print(f"{'Column':<25} | {name1:<15} | {name2:<15}")
        print("-" * 60)
        for col, t1, t2 in mismatches:
            print(f"{col:<25} | {t1:<15} | {t2:<15}")
            
#compare schemas
compare_schemas(sdf_green, sdf_yellow, "Green", "Yellow")
compare_schemas(sdf_green, sdf_hv, "Green", "HV")
# join all datasets
# shape: (737214, 6) + (61018058, 6) + (333726631, 6) -> (395481903, 6)
sdf_all = sdf_green.unionByName(sdf_yellow).unionByName(sdf_hv)
print_shape(sdf_all, "sdf_all")

# check schema
# sdf_all.printSchema()

Schemas match!
Schemas match!
sdf_all shape: (395481903, 6)


In [12]:
# =============================================================================
# 1. Load construction closures and explode date ranges into hourly slots
# =============================================================================
def build_closures_hourly(closures_csv_path: str):
    """Turn each (location_id, start_time, end_time) closure into one row
    per hour it was active, then count concurrent closures per
    (date, hour, location_id).
    """
    raw_closures = (
        spark.read.csv(closures_csv_path, header=True, inferSchema=True)
        .withColumn("start_time", F.col("start_time").cast("timestamp"))
        .withColumn("end_time", F.col("end_time").cast("timestamp"))
        .withColumn("location_id", F.col("location_id").cast("int"))
    )
 
    closures_hourly = (
        raw_closures
        .withColumn("start_hour", F.date_trunc("hour", "start_time"))
        .withColumn("end_hour", F.date_trunc("hour", "end_time"))
        .withColumn(
            "hourly_ts",
            F.explode(F.sequence(F.col("start_hour"), F.col("end_hour"), F.expr("interval 1 hour"))),
        )
        .withColumn("date", F.to_date("hourly_ts"))
        .withColumn("hour", F.hour("hourly_ts"))
        .groupBy("date", "hour", "location_id")
        .agg(F.count("*").alias("construction_count"))
    )
    return closures_hourly
 
 
# =============================================================================
# 2. Build the scaffold: every (date, hour, location_id, taxi_type) that
#    should exist in the final table.
#
#    IMPORTANT: the scaffold must be derived from the RIDE data's own
#    footprint of (date, hour, location_id) — not from closures_hourly.
#    Building it from closures_hourly would restrict every output row to
#    location-hours that already had an active closure, making it
#    impossible for construction_count to ever be 0.
# =============================================================================
def build_scaffold(rides_df, location_col: str, datetime_col: str):
    """Every distinct (date, hour, location_id) that appears in the ride
    data, crossed with every taxi_type, so no combination is missing.
    """
    ride_hours = (
        rides_df
        .withColumn("date", F.to_date(datetime_col))
        .withColumn("hour", F.hour(datetime_col))
        .select("date", "hour", F.col(location_col).alias("location_id"))
        .distinct()
    )
    taxi_types = rides_df.select("taxi_type").distinct()
    return ride_hours.crossJoin(taxi_types)
 
 
# =============================================================================
# 3. Aggregate rides onto the scaffold and attach closure counts
# =============================================================================
def aggregate_rides(rides_df, closures_hourly, location_col: str, datetime_col: str, name: str):
    """Aggregate ride demand + avg metrics by (date, hour, location, taxi_type),
    then left-join onto the full scaffold so every combination is present,
    and left-join closure counts so location-hours with no closure get 0.
    """
    scaffold = build_scaffold(rides_df, location_col, datetime_col)
    print_shape(scaffold, "scaffold")
 
    ride_agg = (
        rides_df
        .withColumn("date", F.to_date(datetime_col))
        .withColumn("hour", F.hour(datetime_col))
        .groupBy("date", "hour", F.col(location_col).alias("location_id"), "taxi_type")
        .agg(
            F.count("*").alias("ride_count"),
            F.avg("total_amount").alias("avg_total_amount"),
            F.avg("trip_duration").alias("avg_trip_duration"),
        )
    )
 
    result = (
        scaffold
        .join(ride_agg, on=["date", "hour", "location_id", "taxi_type"], how="left")
        .join(closures_hourly, on=["date", "hour", "location_id"], how="left")
        .fillna(0, subset=["ride_count", "construction_count"])
    )
    print_shape(result, name)
    return result
 
 
# =============================================================================
# 4. Small helper for sanity-checking row counts as the pipeline runs
# =============================================================================
def print_shape(df, name: str):
    print(f"{name}: {df.count():,} rows")
 
 
# =============================================================================
# 5. Write output as a single parquet file, atomically replacing any
#    previous version
# =============================================================================
def write_single_parquet(df, output_dir: str, filename: str):
    final_path = os.path.join(output_dir, filename)
    tmp_path = os.path.join(output_dir, f"_{filename}_tmp")
 
    df.coalesce(1).write.mode("overwrite").parquet(tmp_path)
 
    part_files = glob.glob(os.path.join(tmp_path, "part-*.parquet"))
    if not part_files:
        raise RuntimeError(f"No part file found in {tmp_path} — write may have failed.")
 
    if os.path.exists(final_path):
        os.remove(final_path)
    shutil.move(part_files[0], final_path)
    shutil.rmtree(tmp_path)
 
    print(f"Wrote {final_path}")
 
 
# =============================================================================
# MAIN
# =============================================================================
if __name__ == "__main__":
    CLOSURES_CSV = "../../data/processed/construction_closures.csv"
    OUTPUT_DIR = "../../data/processed"
 
    # `spark` and `sdf_all` (all taxi trips, with columns pu_location_id,
    # pickup_datetime, taxi_type, total_amount, trip_duration) are assumed
    # to already exist in the notebook session, as in the original script.
 
    closures_hourly = build_closures_hourly(CLOSURES_CSV)
 
    pu_agg = aggregate_rides(
        rides_df=sdf_all,
        closures_hourly=closures_hourly,
        location_col="pu_location_id",
        datetime_col="pickup_datetime",
        name="pu_agg",
    )
 
    # --- Sanity check: confirm 0-closure rows actually exist before writing ---
    zero_closure_rows = pu_agg.filter(F.col("construction_count") == 0).count()
    print(f"Rows with construction_count == 0: {zero_closure_rows:,}")
    if zero_closure_rows == 0:
        raise ValueError(
            "No rows with construction_count == 0 were produced. The scaffold "
            "is likely still being built from closures_hourly instead of from "
            "the ride data's own (date, hour, location_id) footprint."
        )
 
    write_single_parquet(pu_agg, OUTPUT_DIR, "pu_agg.parquet")

scaffold: 9,353,928 rows
pu_agg: 9,353,928 rows
Rows with construction_count == 0: 4,119,846
Wrote ../../data/processed\pu_agg.parquet


In [ ]:
# Load pu_agg parquet and convert to a single CSV file
def write_single_csv(df, output_dir: str, filename: str):
    final_path = os.path.join(output_dir, filename)
    tmp_path = os.path.join(output_dir, f"_{filename}_tmp")

    df.coalesce(1).write.mode("overwrite").option("header", True).csv(tmp_path)

    part_files = glob.glob(os.path.join(tmp_path, "part-*.csv"))
    if not part_files:
        raise RuntimeError(f"No part file found in {tmp_path} — write may have failed.")

    if os.path.exists(final_path):
        os.remove(final_path)
    shutil.move(part_files[0], final_path)
    shutil.rmtree(tmp_path)

    print(f"Wrote {final_path}")


OUTPUT_DIR = "../../data/processed"
pu_agg_loaded = spark.read.parquet(os.path.join(OUTPUT_DIR, "pu_agg.parquet"))
print_shape(pu_agg_loaded, "pu_agg (loaded)")
pu_agg_loaded.printSchema()

write_single_csv(pu_agg_loaded, OUTPUT_DIR, "pu_agg.csv")